## Step 1 :- Project planning & Problem Framing
---

### 1. What is Data Analysis?
Ans.

Data Analysis is the process of inspecting, cleaning, transforming, and modeling data with the goal of discovering useful information, drawing conclusions, and supporting decision-making. It is a crucial step before building any machine learning or AI system, as it helps us understand the story hidden inside data.

---

### 2. Steps in a Data Science Project.
Ans.

1.   Define a Problem.
2.   Data Collection.
3.   Handling Missing Values.
4.   Find Patterns and Trends.
5.   Data Cleanning.
6.   Data Visualization.
7.   Build Ml Models.
8.   Maintain Model.

---
    

### 3. How a frame this dataset as a Machine Learning Problem ?
Ans.

Framing a Machine Learning Problem means translating a real-world business or research question into a well-defined ML task. It involves identifying:
- The type of problem (classification, regression, clustering, etc.)
- The input features (X)
- The target variable (y)

---




## Step 2 :- Data Import and Understanding
---

In [ ]:
import requests
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler, Normalizer

In [ ]:
url = "https://dummyjson.com/users"
response = requests.get(url)

api_data = response.json()
users = pd.DataFrame(api_data['users'])

print("\nAPI Users Data:")
users.head()

In [ ]:
conn = sqlite3.connect("products.db")

with open("products.sql", "r") as file:
    conn.executescript(file.read())

products = pd.read_sql("SELECT * FROM products", conn)

In [ ]:
transactions = pd.read_json("transactions.json")

transactions.head()

In [ ]:
customers = pd.read_csv("customers.csv")

customers.head()

In [ ]:
data = pd.merge(customers, transactions, on="customer_id", how="inner")
data = pd.merge(data, products, on="product_id", how="inner")

data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

## Step 3 :- Exploratory Data Analysis(EDA)
---

In [ ]:
data.columns = data.columns.str.strip().str.lower()
print("Columns:", data.columns)
if 'purchased' not in data.columns:
    if 'quantity' in data.columns:
        purchased_list = []
        for x in data['quantity']:
            if x > 0:
                purchased_list.append(1)
            else:
                purchased_list.append(0)
        data['purchased'] = purchased_list
    else:
        data['purchased'] = 1
numeric_data = data.select_dtypes(include=['int64', 'float64'])

#### Univarte Analysis :-

####(A) Histogram :-

In [ ]:
for col in numeric_data.columns:
    plt.figure()
    numeric_data[col].hist(bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

### (B) Skewed

In [ ]:
print(numeric_data.skew())

### (C) Pandas Profiling

In [ ]:
profile = ProfileReport(data)
profile.to_file("report.html")

### Bivariate Analysis :-

### (A) Correlation Matrix

In [ ]:
plt.figure()
sns.heatmap(numeric_data.corr(), annot=True)
plt.title("Correlation Matrix")
plt.show()

### (B) Numerical vs Target

In [ ]:
valid_cols = []
for col in numeric_data.columns:
    if col != 'purchased':
        valid_cols.append(col)
first_col = valid_cols[0]
plt.figure()
sns.boxplot(x="purchased", y=first_col, data=data)
plt.title(first_col + " vs Purchase")
plt.show()

# Step 4 :- Handling Missing Data
---

## There are no missing values in given data.
---

# Step 5 :- Outiler Detection & Handling
---

### Z - Score:-

In [ ]:
num_cols = data.select_dtypes(include=['int64', 'float64']).columns

z_data = data.copy()

for col in num_cols:
    z_scores = np.abs(stats.zscore(z_data[col]))
    z_data = z_data[(z_scores < 3)]

print(z_data.shape)

### IQR Method:-

In [ ]:
iqr_data = data.copy()

for col in num_cols:
    Q1 = iqr_data[col].quantile(0.25)
    Q3 = iqr_data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    iqr_data = iqr_data[(iqr_data[col] >= lower) & (iqr_data[col] <= upper)]

print(iqr_data.shape)

### Percentile Method:-

In [ ]:
percetile_data = data.copy()

for col in num_cols:
    lower_bound = percetile_data[col].quantile(0.05)
    upper_bound = percetile_data[col].quantile(0.95)

    percetile_data = percetile_data[(percetile_data[col] >= lower_bound) & (percetile_data[col] <= upper_bound)]

print(percetile_data.shape)

# Step 6 :- Handling Mixed & Date/Time Variables
---

In [ ]:
data

In [ ]:
data.columns = data.columns.str.strip().str.lower()

In [ ]:
for col in ['signup_date', 'last_purchase_date']:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors='coerce')

In [ ]:
data.dtypes

# Step 7 :- Encoding Caterogical Data
---

In [ ]:
data.columns = data.columns.str.strip().str.lower()
cat_cols = data.select_dtypes(include=['object']).columns

### Lable Encoding

In [ ]:
for col in cat_cols:
    if data[col].nunique() == 2:
        data[col] = le.fit_transform(data[col])
        print(col)


### One-Hot Encoding

In [ ]:
data = pd.get_dummies(data, columns=cat_cols, drop_first=True)

### Ordinal Encoding

In [ ]:
if 'education' in data.columns:
    education_order = {'high school': 1,'bachelor': 2,'master': 3,'phd': 4}
    data['education'] = data['education'].map(education_order)

### Numerical Encoding

In [ ]:
if 'annual_income' in data.columns:
    data['income_group'] = pd.cut(data['annual_income'],bins=3,labels=['low', 'medium', 'high'])
    data['income_group'] = LabelEncoder().fit_transform(data['income_group'])

In [ ]:
data.dtypes

# Step 8 :- Feature Scaling
---

In [ ]:
num_cols = data.select_dtypes(include=['int64', 'float64']).columns

### 1. StandardScaler

In [ ]:
standard_data = data.copy()
scaler = StandardScaler()
standard_data[num_cols] = scaler.fit_transform(standard_data[num_cols])

### 2. MinMaxScaler

In [ ]:
minmax_data = data.copy()
scaler = MinMaxScaler()
minmax_data[num_cols] = scaler.fit_transform(minmax_data[num_cols])

### 3. MaxAbsScaler

In [ ]:
maxabs_data = data.copy()
scaler = MaxAbsScaler()
maxabs_data[num_cols] = scaler.fit_transform(maxabs_data[num_cols])

### 4. RobustScaler

In [ ]:
robust_data = data.copy()
scaler = RobustScaler()
robust_data[num_cols] = scaler.fit_transform(robust_data[num_cols])

### 5. Normalizer

In [ ]:
normalizer_data = data.copy()
scaler = Normalizer()
normalizer_data[num_cols] = scaler.fit_transform(normalizer_data[num_cols])

# Save File:-

In [ ]:
data.to_csv("Processed_customer_data.csv", index=False)